In [1]:
%pip install numpy pandas tensorflow scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import MinMaxScaler

df = pd.read_csv("DailyDelhiClimateTest.csv")
series = df['meantemp'].values.reshape(-1, 1)

scaler = MinMaxScaler()
series_scaled = scaler.fit_transform(series).flatten()

window_size = 30
horizon = 1

def make_dataset(series, window_size, horizon):
    X, y = [], []
    for i in range(len(series) - window_size - horizon + 1):
        X.append(series[i:i+window_size])
        y.append(series[i+window_size:i+window_size+horizon])
    return np.array(X), np.array(y)

X, y = make_dataset(series_scaled, window_size, horizon)
X = X[..., None]

split = int(0.8 * len(X))
X_train, y_train = X[:split], y[:split]
X_test, y_test   = X[split:], y[split:]

model = keras.Sequential([
    layers.Input(shape=(window_size, 1)),
    layers.LSTM(64),
    layers.Dense(horizon)
])

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="mse",
    metrics=["mae"]
)

history = model.fit(X_train, y_train, epochs=20, batch_size=32, validation_split=0.2)

test_loss, test_mae = model.evaluate(X_test, y_test, verbose=0)
print("Test MAE:", test_mae)

Epoch 1/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 309ms/step - loss: 0.1711 - mae: 0.3867 - val_loss: 0.5153 - val_mae: 0.7155
Epoch 2/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 113ms/step - loss: 0.1232 - mae: 0.3207 - val_loss: 0.3618 - val_mae: 0.5987
Epoch 3/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - loss: 0.0809 - mae: 0.2500 - val_loss: 0.2254 - val_mae: 0.4710
Epoch 4/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step - loss: 0.0454 - mae: 0.1727 - val_loss: 0.1095 - val_mae: 0.3244
Epoch 5/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - loss: 0.0198 - mae: 0.1011 - val_loss: 0.0288 - val_mae: 0.1597
Epoch 6/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - loss: 0.0153 - mae: 0.1111 - val_loss: 0.0073 - val_mae: 0.0643
Epoch 7/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - loss: 0.0241 - mae: 0.1312 - val_loss: 0.0085 - val_mae: 0.0688
Epoch 8/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - loss: 0.0254 - mae: 0.1347 - val_loss: 0.0072 - val_mae: 0.0639
Epoch 9/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - loss: 0.0179 - mae: 0.1168 

### Save and package the model

In [4]:
import os
import tarfile
import sagemaker
from sagemaker.tensorflow import TensorFlowModel

# Keras 3 requires .export() for TensorFlow Serving compatibility
model_version = '1'
export_dir = os.path.join('export/Servo', model_version)

# Use export instead of save
model.export(export_dir)

# Create model.tar.gz
with tarfile.open('model.tar.gz', mode='w:gz') as archive:
    # We archive the 'export' directory which now contains the SavedModel
    archive.add('export', recursive=True)

# Upload to S3
sagemaker_session = sagemaker.Session()
bucket = sagemaker_session.default_bucket()
prefix = 'sagemaker/delhi-climate-lstm'
model_data = sagemaker_session.upload_data(path='model.tar.gz', bucket=bucket, key_prefix=prefix)

INFO:tensorflow:Assets written to: export/Servo/1/assets


INFO:tensorflow:Assets written to: export/Servo/1/assets


Saved artifact at 'export/Servo/1'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 30, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  140025968288848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140025968289808: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140025968289616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140025968290768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140025968290000: TensorSpec(shape=(), dtype=tf.resource, name=None)


### Deploy to serverless endpoint

In [5]:
from sagemaker.tensorflow import TensorFlowModel
from sagemaker.serverless import ServerlessInferenceConfig

# Get the execution role for the notebook
role = sagemaker.get_execution_role()

# Define the SageMaker Model
# We use version 2.16 to ensure compatibility with model.export()
sm_model = TensorFlowModel(
    model_data=model_data,
    role=role,
    framework_version='2.16',
    sagemaker_session=sagemaker_session
)

# Configure Serverless: 2048 MB is usually the sweet spot for LSTM memory/CPU ratio
serverless_config = ServerlessInferenceConfig(
    memory_size_in_mb=2048,
    max_concurrency=5
)

# Deploy the endpoint
# This will take approximately 2-4 minutes to provision
predictor = sm_model.deploy(
    serverless_inference_config=serverless_config,
    endpoint_name='delhi-climate-serverless-ep'
)

--!

### Test endpoint

In [6]:
import boto3
client = boto3.client('sagemaker')

response = client.describe_endpoint(EndpointName='delhi-climate-serverless-ep')
print(f"Endpoint Status: {response['EndpointStatus']}")

Endpoint Status: InService


### Test the endpoint

In [7]:
import numpy as np

# Prepare input: X_test is (N, 30, 1). We take the first sample.
# TensorFlow Serving expects the data wrapped in an 'instances' key.
sample_input = {"instances": X_test[:1].tolist()}

# Invoke the endpoint
# Note: The first call may take longer due to 'Cold Start'
response = predictor.predict(sample_input)

# Extract and scale back the results
# response['predictions'] will be a list of lists [[val]]
predicted_val_scaled = np.array(response['predictions'])
predicted_temp = scaler.inverse_transform(predicted_val_scaled)

# Get actual value for comparison
actual_temp = scaler.inverse_transform(y_test[:1])

print(f"Predicted Temperature: {predicted_temp[0][0]:.2f}°C")
print(f"Actual Temperature:    {actual_temp[0][0]:.2f}°C")

Predicted Temperature: 29.00°C
Actual Temperature:    25.62°C
